In [0]:
create or refresh streaming table bronze_customers
comment 'Raw Customer data from source'
tblproperties('quality'='bronze')
as select *,
  _metadata.file_path as file_path,
    current_timestamp as ingestion_timestamp
from cloud_files(
    '/Volumes/circuitbox/landing/operational_data/customers',
    'json',
    map('cloudFiles.inferColumnTypes','true')
);

Streaing table for clean Customer Silver layer

In [0]:
create or refresh streaming table silver_customers_clean (
    constraint valid_customer_id expect(customer_id is not null) on violation fail update,
    constraint valid_customer_name expect(customer_name is not null) on violation drop row,
    constraint valid_telephone expect(length(telephone)>=10),
    constraint valid_email expect(email is not null),
    constraint valid_date_of_birth expect(date_of_birth >='1920-01-01')
)
comment 'Cleaned Customer data'
tblproperties('quality'='silver')
as
select customer_id,
    customer_name,
    cast(date_of_birth as date) as date_of_birth,
    telephone,
    email,
    cast(created_date as date) as created_date
from stream(bronze_customers)

Streaming Silver table for Cusomers

In [0]:
create or refresh streaming table silver_customers
comment 'SCD Type 1 Customer data'
tblproperties('quality'='silver');

In [0]:
APPLY CHANGES into silver_customers
from stream(silver_customers_clean)
keys(customer_id)
SEQUENCE BY created_date
STORED AS SCD TYPE 1;